<a href="https://colab.research.google.com/github/12halima/Transport_Recommander/blob/main/Transport_Recommander/notebooks/02_load_gtfs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import pandas as pd

GTFS_CLEAN = "/content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN"   # adapte si besoin

all_agencies = []  # liste pour stocker tous les df

# Parcourir les sous-dossiers
for subfolder in sorted(os.listdir(GTFS_CLEAN)):
    folder_path = os.path.join(GTFS_CLEAN, subfolder)

    # vérifier que c'est un dossier
    if not os.path.isdir(folder_path):
        continue

    agency_path = os.path.join(folder_path, "agency.txt")

    # vérifier que le fichier existe
    if os.path.exists(agency_path):
        try:
            df = pd.read_csv(agency_path, encoding="utf-8", on_bad_lines="skip")
            df["source_folder"] = subfolder  # pour savoir d'où vient le fichier
            all_agencies.append(df)
            print(f"📥 agency.txt chargé depuis : {subfolder}")
        except Exception as e:
            print(f"⚠️ Erreur lecture dans {subfolder}: {e}")
    else:
        print(f"❌ Aucun agency.txt dans : {subfolder}")


📥 agency.txt chargé depuis : AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)
📥 agency.txt chargé depuis : ALSA buses
📥 agency.txt chargé depuis : AUCORSA (Autobuses de Córdoba S.A.)
📥 agency.txt chargé depuis : AUTNA SL
📥 agency.txt chargé depuis : Alavabus
📥 agency.txt chargé depuis : Alvarez Travelers Coaches
📥 agency.txt chargé depuis : Ancebus
📥 agency.txt chargé depuis : Auif Irunbus (Lurraldebus)
📥 agency.txt chargé depuis : Autocares Baraza (Baraza Coaches)
📥 agency.txt chargé depuis : Autocares Rías Baixas (Rías Baixas Coaches)
📥 agency.txt chargé depuis : Autocorb Coaches
📥 agency.txt chargé depuis : Autoridad de Transporte Metropolitano del Area de Barcelona (ATM) Buses and trains in Catalonia (full version)
📥 agency.txt chargé depuis : Avanza Grupo (Ávila city bus)
📥 agency.txt chargé depuis : Avanza Grupo (Huesca city bus)
📥 agency.txt chargé depuis : Avanza Grupo (Mataró city bus)
📥 agency.txt chargé depuis : Avanza Grupo (Segovia city bus)
📥 agency.txt chargé depuis 

In [3]:
if all_agencies:
    agencies_df = pd.concat(all_agencies, ignore_index=True)
    print("\n📊 Total lignes agence :", len(agencies_df))
else:
    print("❌ Aucun fichier agency.txt trouvé.")



📊 Total lignes agence : 2372


In [4]:
# Afficher les 10 premières lignes
agencies_df.head(10)


,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone,source_folder,agency_fare_url,agency_email,agency_branding_url
0,a-28005890,AISA,https://www.aisa-grupo.com/,Europe/Madrid,es,34918752018,AISA (Bus Madrid-Aranda de Duero-Burgo de Osma),NaN,NaN,NaN
1,300,ALSA,http://www.alsa.es,Europe/Madrid,es,902422242,ALSA buses,NaN,NaN,NaN
2,Aucorsa,Autobuses de Córdoba - S.A.,https://www.aucorsa.es,Europe/Madrid,es,957764676,AUCORSA (Autobuses de Córdoba S.A.),https://www.aucorsa.es/gestion_y_recarga_onlin...,NaN,NaN
3,autna,AUTNA,https://www.autna.com/,Europe/Madrid,es,0034 986 288 030,AUTNA SL,NaN,NaN,NaN
4,1,Álavabus,https://alavabus.eus/es,Europe/Madrid,es,945182060,Alavabus,https://alavabus.eus/es/tarifas-y-otros,alavabus@araba.eus,NaN
5,2,Transporte Comarcal,https://alavabus.eus/es,Europe/Madrid,es,945182060,Alavabus,https://alavabus.eus/es/tarifas-y-otros,alavabus@araba.eus,NaN
6,AUTOS ALVAREZ,AUTOS ALVAREZ DE VIAJEROS SL,http://vulpeti.com/moderniza/B49122856.zip,Europe/Madrid,es,980 62 05 01,Alvarez Travelers Coaches,NaN,NaN,NaN
7,ANCEBUS,"ANCEBUS, SL",http://www.ancebus.com/,Europe/Madrid,es,"923 480 575, 695 345 243",Ancebus,NaN,NaN,NaN
8,13,Auif,http://www.auif.es/,Europe/Madrid,es,943633111,Auif Irunbus (Lurraldebus),https://www.mugi.eus/index.php/es/tarjetas/tar...,NaN,NaN
9,1,Autocares Baraza,http://www.autocaresbaraza.com,Europe/Madrid,es,950390311,Autocares Baraza (Baraza Coaches),NaN,NaN,NaN


In [5]:
required_cols = ["agency_id", "agency_name", "agency_timezone"]

# Compter les valeurs null ou NaN
agencies_df[required_cols].isna().sum()


,0
agency_id,1
agency_name,0
agency_timezone,0


In [6]:
for col in required_cols:
    nan_count = agencies_df[col].isna().sum()
    empty_count = (agencies_df[col].astype(str).str.strip() == "").sum()

    print(f"🔹 {col} : NaN = {nan_count}, vides = {empty_count}, TOTAL = {nan_count + empty_count}")


🔹 agency_id : NaN = 1, vides = 0, TOTAL = 1
🔹 agency_name : NaN = 0, vides = 0, TOTAL = 0
🔹 agency_timezone : NaN = 0, vides = 0, TOTAL = 0


In [7]:
# Colonnes obligatoires
required_cols = ["agency_id", "agency_name", "agency_timezone"]

# Supprimer les lignes invalides
clean_df = agencies_df.dropna(subset=required_cols)

# Supprimer aussi les lignes vides ""
for col in required_cols:
    clean_df = clean_df[clean_df[col].astype(str).str.strip() != ""]


In [8]:
deleted_rows = len(agencies_df) - len(clean_df)
print(f"🚮 Lignes supprimées : {deleted_rows}")
print(f"📊 Lignes restantes : {len(clean_df)}")


🚮 Lignes supprimées : 1
📊 Lignes restantes : 2371


In [9]:
extra_cols = [
    "agency_url",
    "agency_lang",
    "agency_phone",
    "agency_fare_url",
    "agency_email",
    "agency_branding_url",
    "source_folder"
]

print("📊 Vérification des valeurs null / vides dans les colonnes supplémentaires\n")

for col in extra_cols:
    nan_count = agencies_df[col].isna().sum()
    empty_count = (agencies_df[col].astype(str).str.strip() == "").sum()
    total = nan_count + empty_count

    print(f"🔹 {col} : NaN = {nan_count}, vides = {empty_count}, TOTAL = {total}")


📊 Vérification des valeurs null / vides dans les colonnes supplémentaires

🔹 agency_url : NaN = 0, vides = 0, TOTAL = 0
🔹 agency_lang : NaN = 2040, vides = 0, TOTAL = 2040
🔹 agency_phone : NaN = 1780, vides = 0, TOTAL = 1780
🔹 agency_fare_url : NaN = 2302, vides = 0, TOTAL = 2302
🔹 agency_email : NaN = 2323, vides = 0, TOTAL = 2323
🔹 agency_branding_url : NaN = 2371, vides = 0, TOTAL = 2371
🔹 source_folder : NaN = 0, vides = 0, TOTAL = 0


In [10]:
# 📌 Colonnes à supprimer (inutile pour ton projet)
cols_to_drop = ["agency_branding_url", "agency_fare_url", "agency_email"]

# Supprimer les colonnes
agencies_df = agencies_df.drop(columns=cols_to_drop)

# Vérifier le résultat
print("Colonnes restantes :", agencies_df.columns.tolist())


Colonnes restantes : ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone', 'source_folder']


In [11]:
# 🔹 Colonnes facultatives
optional_cols = ["agency_lang", "agency_phone", "agency_url"]

# 1️⃣ Remplir les valeurs vides ou NaN
# - agency_lang : mettre 'es' par défaut si vide
# - agency_phone et agency_url : laisser NaN
import numpy as np

agencies_df["agency_lang"] = agencies_df["agency_lang"].fillna("es")
agencies_df["agency_phone"] = agencies_df["agency_phone"].replace("", np.nan)
agencies_df["agency_url"] = agencies_df["agency_url"].replace("", np.nan)

# 2️⃣ Nettoyer les numéros de téléphone
# Supprimer tous les caractères non numériques sauf '+' au début
import re

def clean_phone(phone):
    if pd.isna(phone):
        return np.nan
    phone = str(phone).strip()
    phone = re.sub(r"[^\d+]", "", phone)
    return phone

agencies_df["agency_phone"] = agencies_df["agency_phone"].apply(clean_phone)

# 3️⃣ Vérifier le résultat
print(agencies_df.head())


    agency_id                  agency_name                   agency_url  \
0  a-28005890                         AISA  https://www.aisa-grupo.com/   
1         300                         ALSA           http://www.alsa.es   
2     Aucorsa  Autobuses de Córdoba - S.A.       https://www.aucorsa.es   
3       autna                        AUTNA       https://www.autna.com/   
4           1                     Álavabus      https://alavabus.eus/es   

  agency_timezone agency_lang   agency_phone  \
0   Europe/Madrid          es    34918752018   
1   Europe/Madrid          es      902422242   
2   Europe/Madrid          es      957764676   
3   Europe/Madrid          es  0034986288030   
4   Europe/Madrid          es      945182060   

                                     source_folder  
0  AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)  
1                                       ALSA buses  
2             AUCORSA (Autobuses de Córdoba S.A.)  
3                                         AUTNA 

In [12]:
# 🔹 Nettoyage avancé des colonnes obligatoires
def clean_obligatory_columns(df):
    # agency_id : minuscule, strip
    df["agency_id"] = df["agency_id"].astype(str).str.strip().str.lower()

    # agency_name : strip, optionnel title()
    df["agency_name"] = df["agency_name"].astype(str).str.strip().str.title().str.lower()

    # agency_timezone : strip, minuscule
    df["agency_timezone"] = df["agency_timezone"].astype(str).str.strip().str.lower()

    return df

agencies_df = clean_obligatory_columns(agencies_df)

# Vérifier le résultat
print(agencies_df.head())


    agency_id                  agency_name                   agency_url  \
0  a-28005890                         aisa  https://www.aisa-grupo.com/   
1         300                         alsa           http://www.alsa.es   
2     aucorsa  autobuses de córdoba - s.a.       https://www.aucorsa.es   
3       autna                        autna       https://www.autna.com/   
4           1                     álavabus      https://alavabus.eus/es   

  agency_timezone agency_lang   agency_phone  \
0   europe/madrid          es    34918752018   
1   europe/madrid          es      902422242   
2   europe/madrid          es      957764676   
3   europe/madrid          es  0034986288030   
4   europe/madrid          es      945182060   

                                     source_folder  
0  AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)  
1                                       ALSA buses  
2             AUCORSA (Autobuses de Córdoba S.A.)  
3                                         AUTNA 

In [13]:
# Convertir toutes les langues en minuscules
agencies_df['agency_lang'] = agencies_df['agency_lang'].str.lower()


In [15]:
cols_order = ['agency_id', 'agency_name', 'agency_timezone',
              'agency_lang', 'agency_phone', 'agency_url', 'source_folder']
agencies_df = agencies_df[cols_order]


In [16]:
agencies_df["agency_id"] = agencies_df["agency_id"].astype(str)


In [17]:
# Garder uniquement les chiffres
agencies_df["agency_phone"] = agencies_df["agency_phone"].astype(str).str.replace(r"\D", "", regex=True)

# Garder seulement les numéros de 9 chiffres
agencies_df["agency_phone"] = agencies_df["agency_phone"].apply(
    lambda x: x if len(x) == 9 else None
)


In [18]:
import numpy as np

# 1. Convertir en string et enlever tout sauf les chiffres
agencies_df["agency_phone"] = agencies_df["agency_phone"].astype(str).str.replace(r"\D", "", regex=True)

# 2. Remplacer "nan", "" et "None" par NaN réel
agencies_df["agency_phone"].replace(["", "nan", "None"], np.nan, inplace=True)

# 3. Garder seulement les numéros de 9 chiffres, sinon NaN
agencies_df["agency_phone"] = agencies_df["agency_phone"].apply(
    lambda x: x if isinstance(x, str) and len(x) == 9 else np.nan
)


/tmp/ipython-input-1045780983.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  agencies_df["agency_phone"].replace(["", "nan", "None"], np.nan, inplace=True)


In [19]:
# Repérer les doublons exacts
doublons = agencies_df[agencies_df.duplicated(keep=False)]

# Afficher les lignes doublées
print(doublons)


Empty DataFrame
Columns: [agency_id, agency_name, agency_timezone, agency_lang, agency_phone, agency_url, source_folder]
Index: []


In [20]:
# Afficher les 10 premières lignes
agencies_df.head(10)


,agency_id,agency_name,agency_timezone,agency_lang,agency_phone,agency_url,source_folder
0,a-28005890,aisa,europe/madrid,es,NaN,https://www.aisa-grupo.com/,AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)
1,300,alsa,europe/madrid,es,902422242,http://www.alsa.es,ALSA buses
2,aucorsa,autobuses de córdoba - s.a.,europe/madrid,es,957764676,https://www.aucorsa.es,AUCORSA (Autobuses de Córdoba S.A.)
3,autna,autna,europe/madrid,es,NaN,https://www.autna.com/,AUTNA SL
4,1,álavabus,europe/madrid,es,945182060,https://alavabus.eus/es,Alavabus
5,2,transporte comarcal,europe/madrid,es,945182060,https://alavabus.eus/es,Alavabus
6,autos alvarez,autos alvarez de viajeros sl,europe/madrid,es,980620501,http://vulpeti.com/moderniza/B49122856.zip,Alvarez Travelers Coaches
7,ancebus,"ancebus, sl",europe/madrid,es,NaN,http://www.ancebus.com/,Ancebus
8,13,auif,europe/madrid,es,943633111,http://www.auif.es/,Auif Irunbus (Lurraldebus)
9,1,autocares baraza,europe/madrid,es,950390311,http://www.autocaresbaraza.com,Autocares Baraza (Baraza Coaches)


In [21]:
import os

# Dossier principal contenant les sous-dossiers
main_folder = "/content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN"  # mettre ton chemin exact

# Parcourir toutes les valeurs uniques de source_folder
for subfolder_name in agencies_df["source_folder"].unique():
    subfolder_path = os.path.join(main_folder, subfolder_name)

    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)

    # Sélectionner uniquement les lignes correspondant à ce sous-dossier
    df_subset = agencies_df[agencies_df["source_folder"] == subfolder_name].copy()

    # Supprimer la colonne source_folder
    if "source_folder" in df_subset.columns:
        df_subset.drop(columns=["source_folder"], inplace=True)

    # Définir le chemin du fichier
    output_file = os.path.join(subfolder_path, "agency_clean.txt")

    # Sauvegarder le dataframe filtré
    df_subset.to_csv(output_file, index=False, sep=",", encoding="utf-8")

    print(f"📥 Fichier créé : {output_file} ({len(df_subset)} lignes)")


📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/ALSA buses/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/AUCORSA (Autobuses de Córdoba S.A.)/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/AUTNA SL/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Alavabus/agency_clean.txt (2 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Alvarez Travelers Coaches/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Ancebus/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Auif Irunbus (Lurraldebus)/agency_clean.txt (1 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Autocares Baraza (Baraza Coaches)/agency_cl

In [22]:
import os
import pandas as pd

# Dossier principal contenant les sous-dossiers
GTFS_CLEAN = "/content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN"  # mettre ton chemin exact

all_stops = []  # liste pour stocker tous les df stops.txt

# Parcourir les sous-dossiers
for subfolder in sorted(os.listdir(GTFS_CLEAN)):
    folder_path = os.path.join(GTFS_CLEAN, subfolder)

    # Vérifier que c'est bien un dossier
    if not os.path.isdir(folder_path):
        continue

    stops_path = os.path.join(folder_path, "stops.txt")

    # Vérifier que le fichier stops.txt existe
    if os.path.exists(stops_path):
        try:
            df = pd.read_csv(stops_path, encoding="utf-8", on_bad_lines="skip")

            # Ajouter la colonne pour savoir d'où vient la ligne
            df["source_folder"] = subfolder

            # Ajouter le dataframe à la liste
            all_stops.append(df)

            print(f"📥 stops.txt chargé depuis : {subfolder} ({len(df)} lignes)")
        except Exception as e:
            print(f"⚠️ Erreur lecture dans {subfolder}: {e}")
    else:
        print(f"❌ Aucun stops.txt dans : {subfolder}")

# Combiner tous les dataframes en un seul si nécessaire
if all_stops:
    stops_df = pd.concat(all_stops, ignore_index=True)
    print(f"\n✅ stops.txt combinés : {len(stops_df)} lignes au total")
else:
    print("\n❌ Aucun stops.txt trouvé dans tous les sous-dossiers")


📥 stops.txt chargé depuis : AISA (Bus Madrid-Aranda de Duero-Burgo de Osma) (36 lignes)
📥 stops.txt chargé depuis : ALSA buses (11470 lignes)
📥 stops.txt chargé depuis : AUCORSA (Autobuses de Córdoba S.A.) (613 lignes)
📥 stops.txt chargé depuis : AUTNA SL (9 lignes)
📥 stops.txt chargé depuis : Alavabus (678 lignes)
📥 stops.txt chargé depuis : Alvarez Travelers Coaches (65 lignes)
📥 stops.txt chargé depuis : Ancebus (26 lignes)
📥 stops.txt chargé depuis : Auif Irunbus (Lurraldebus) (72 lignes)
📥 stops.txt chargé depuis : Autocares Baraza (Baraza Coaches) (50 lignes)
📥 stops.txt chargé depuis : Autocares Rías Baixas (Rías Baixas Coaches) (2549 lignes)
📥 stops.txt chargé depuis : Autocorb Coaches (170 lignes)
📥 stops.txt chargé depuis : Autoridad de Transporte Metropolitano del Area de Barcelona (ATM) Buses and trains in Catalonia (full version) (28410 lignes)
📥 stops.txt chargé depuis : Avanza Grupo (Ávila city bus) (194 lignes)
📥 stops.txt chargé depuis : Avanza Grupo (Huesca city b

/tmp/ipython-input-1243109629.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  stops_df = pd.concat(all_stops, ignore_index=True)


In [23]:
stops_df.sample(10)


,stop_id,stop_code,stop_name,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,source_folder,stop_desc,wheelchair_boarding,tts_stop_name,level_id,platform_code,out_of_service
233932,3100786,NaN,GOURDAN-POLIGNAN - Mairie,43.077658,0.577116,NaN,NaN,0.0,31S00787,NaN,Réseau interurbain liO Occitanie,arrêt commercial,2.0,NaN,NaN,NaN,NaN
206774,MOBIITI:Quay:74817,SHE01,Shenzhen,46.655445,0.362523,NaN,NaN,0.0,MOBIITI:StopPlace:50220,NaN,Nouvelle-Aquitaine Mobilités,P,2.0,NaN,NaN,NaN,NaN
141994,de:08136:6203:0:1,NaN,"Durlangen, Raiba",48.858903,9.795086,NaN,NaN,NaN,NaN,NaN,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,NaN,0.0,NaN,NaN,NaN,NaN
76415,VIC_9964,9964,Pl. F. Moragas,41.923931,2.254987,NaN,NaN,0.0,NaN,NaN,Catalonia Area de Barcelona,NaN,0.0,NaN,NaN,NaN,NaN
243877,130,130,Libra impares (cruce C/ Escorpio),39.476003,-6.395352,NaN,NaN,NaN,NaN,NaN,Vectalia Movilidad (bus de la ville de Cáceres),NaN,NaN,NaN,NaN,NaN,NaN
116740,7042720,7042720,Barrio,42.790960,-8.119290,7042720,NaN,NaN,NaN,NaN,Lázara Coaches,NaN,NaN,NaN,NaN,NaN,NaN
35538,SAA_14028,17003,"SIURANA, CRUILLA (DIR. FIGUERES)",42.220253,3.014031,NaN,NaN,0.0,NaN,NaN,Autoridad de Transporte Metropolitano del Area...,NaN,0.0,NaN,NaN,NaN,NaN
251732,SP19804,1039647,RAPADOIRA LLAS,43.577034,-7.254950,27019-102,NaN,NaN,NaN,NaN,Xunta de Galicia Buses,NaN,NaN,NaN,NaN,NaN,NaN
51873,4324,48085010,ANDER DEUNA (ERMITA) (4324),43.390511,-2.978667,3,NaN,0.0,NaN,Europe/Madrid,Bizkaibus,NaN,0.0,NaN,0,NaN,NaN
135801,de:08126:10818:0:2,NaN,Kubach,49.240722,9.706413,NaN,NaN,NaN,NaN,NaN,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,NaN,0.0,NaN,NaN,NaN,NaN


In [24]:
import pandas as pd

# Colonnes obligatoires
mandatory_cols = ["stop_id", "stop_name", "stop_lat", "stop_lon"]

# Fonction pour compter les "vraies" valeurs manquantes
def count_missing(series):
    # Convertir en str pour détecter les espaces
    return series.isna().sum() + (series.astype(str).str.strip() == "").sum()

# Appliquer la fonction à chaque colonne obligatoire
missing_summary = {col: count_missing(stops_df[col]) for col in mandatory_cols}

# Afficher le résultat
print("Nombre total de valeurs manquantes, NaN ou vides par colonne obligatoire :\n")
for col, count in missing_summary.items():
    print(f"{col} : {count}")


Nombre total de valeurs manquantes, NaN ou vides par colonne obligatoire :

stop_id : 0
stop_name : 0
stop_lat : 0
stop_lon : 0


In [25]:
# Toutes les colonnes sauf les obligatoires
extra_cols = [col for col in stops_df.columns if col not in ["stop_id", "stop_name", "stop_lat", "stop_lon"]]

# Fonction pour compter les valeurs manquantes, NaN ou vides
def count_missing(series):
    return series.isna().sum() + (series.astype(str).str.strip() == "").sum()

# Nombre total de lignes dans le dataframe
total_rows = len(stops_df)

# Appliquer la fonction à toutes les colonnes supplémentaires et calculer le pourcentage
missing_summary_extra = {col: (count_missing(stops_df[col]), round(count_missing(stops_df[col])/total_rows*100, 2)) for col in extra_cols}

# Afficher le résultat de façon lisible
print("Nombre et pourcentage de valeurs manquantes, NaN ou vides par colonne supplémentaire :\n")
for col, (count, pct) in missing_summary_extra.items():
    print(f"{col} : {count} lignes manquantes ({pct}%)")


Nombre et pourcentage de valeurs manquantes, NaN ou vides par colonne supplémentaire :

stop_code : 145457 lignes manquantes (53.58%)
zone_id : 214207 lignes manquantes (78.9%)
stop_url : 230789 lignes manquantes (85.0%)
location_type : 102736 lignes manquantes (37.84%)
parent_station : 226541 lignes manquantes (83.44%)
stop_timezone : 238362 lignes manquantes (87.79%)
source_folder : 0 lignes manquantes (0.0%)
stop_desc : 209435 lignes manquantes (77.14%)
wheelchair_boarding : 72459 lignes manquantes (26.69%)
tts_stop_name : 270823 lignes manquantes (99.75%)
level_id : 267721 lignes manquantes (98.61%)
platform_code : 263779 lignes manquantes (97.16%)
out_of_service : 266686 lignes manquantes (98.23%)


In [26]:
len(stops_df)


271501

In [27]:
import pandas as pd

# Pourcentage limite pour supprimer la colonne
threshold = 0.7

# Calculer le pourcentage de valeurs manquantes (NaN ou vides) par colonne
def missing_percentage(series):
    total = len(series)
    missing = series.isna().sum() + (series.astype(str).str.strip() == "").sum()
    return missing / total

# Colonnes à supprimer si plus de 70 % manquantes, sauf stop_timezone
cols_to_drop = [col for col in stops_df.columns
                if col != "stop_timezone" and missing_percentage(stops_df[col]) > threshold]

# Supprimer les colonnes
stops_df.drop(columns=cols_to_drop, inplace=True)
print(f"Colonnes supprimées (>70% manquantes) : {cols_to_drop}")

# Remplacer les NaN ou vides dans stop_timezone par 'Europe/Madrid'
stops_df["stop_timezone"] = stops_df["stop_timezone"].replace(to_replace=[None, "", "nan", "NaN"], value="Europe/Madrid")
stops_df["stop_timezone"].fillna("Europe/Madrid", inplace=True)

print("✅ Colonnes nettoyées avec succès")


Colonnes supprimées (>70% manquantes) : ['zone_id', 'stop_url', 'parent_station', 'stop_desc', 'tts_stop_name', 'level_id', 'platform_code', 'out_of_service']
✅ Colonnes nettoyées avec succès


/tmp/ipython-input-884916702.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  stops_df["stop_timezone"].fillna("Europe/Madrid", inplace=True)


In [28]:
stops_df.sample(10)

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_timezone,source_folder,wheelchair_boarding
138210,de:08127:30425:0:1,NaN,Vellberg Alte Post,49.086133,9.880022,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0.0
166395,de:08337:4074:0:1,NaN,Öflingen Brühlstraße,47.599829,7.915819,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0.0
9372,0000247900000002,NaN,Riofrio,42.611601,-5.933955,0.0,Europe/Madrid,ALSA buses,NaN
117154,7037752,7037752,Figueroa,42.543190,-8.393900,NaN,Europe/Madrid,Lázara Coaches,NaN
258868,SP28464,1040290,GUISALLE,42.715926,-7.368589,NaN,Europe/Madrid,Xunta de Galicia Buses,NaN
106399,PF25120110,NaN,"Col·legi Claver (N-240, pk 107,9) (A)",41.688154,0.490587,0.0,Europe/Madrid,Generalitat of Catalonia (Intercity bus),NaN
242927,792dc8e4-6a4b-483b-aff3-82bc58abeac0,660,Avda. Ilustración - Dílar,37.149141,-3.601171,0.0,Europe/Madrid,TranspRober,2.0
88785,par_8_11194,11194,AV.DON JUAN BORBÓN-POLIDEPORTIVO,40.319149,-3.716955,0.0,Europe/Madrid,Consorcio Regional de Transportes de Madrid CR...,2.0
93941,par_8_13094,13094,CTRA.M313-LAS CABRIZAS,40.215420,-3.460803,0.0,Europe/Madrid,Consorcio Regional de Transportes de Madrid CR...,2.0
263638,SP34613,1047482,A MAESTRANZA,43.371097,-8.390813,NaN,Europe/Madrid,Xunta de Galicia Buses,NaN


In [29]:
stops_df["wheelchair_boarding"].value_counts(dropna=False)


,count
wheelchair_boarding,
0.0,130956
NaN,72459
2.0,48945
1.0,19141


In [30]:
stops_df["wheelchair_boarding"] = stops_df["wheelchair_boarding"].fillna(0)


In [31]:
stops_df["wheelchair_boarding"].value_counts(dropna=False)

,count
wheelchair_boarding,
0.0,203415
2.0,48945
1.0,19141


In [32]:
stops_df["wheelchair_boarding"] = stops_df["wheelchair_boarding"].astype(int)


In [33]:
# Normaliser stop_id en minuscules
stops_df['stop_id'] = stops_df['stop_id'].astype(str).str.lower()


In [34]:
# Mettre en minuscules et nettoyer les espaces multiples
stops_df["stop_name"] = stops_df["stop_name"].astype(str)  # s'assurer que c'est du texte
stops_df["stop_name"] = stops_df["stop_name"].str.lower()  # tout en minuscules
stops_df["stop_name"] = stops_df["stop_name"].str.strip()  # enlever espaces au début et à la fin
stops_df["stop_name"] = stops_df["stop_name"].str.replace(r"\s+", " ", regex=True)  # remplacer plusieurs espaces par un seul


In [36]:
stops_df["stop_id"] = stops_df["stop_id"].str.lower()

In [37]:
stops_df["stop_id"] = stops_df["stop_id"].str.strip()

In [38]:
stops_df.sample(10)

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_timezone,source_folder,wheelchair_boarding
88302,par_8_07147,7147,av.zarauz-pºazpeitia,40.419655,-3.527502,0.0,Europe/Madrid,Consorcio Regional de Transportes de Madrid CR...,2
61342,gen_pf08176001,NaN,sant andreu de pujalt,41.717682,1.422436,0.0,Europe/Madrid,Catalonia Area de Barcelona,0
116164,250,250,costa da unión,43.361823,-8.409607,NaN,Europe/Madrid,La Coruña Tram Company SA,0
148821,de:08225:7147:0:riw,NaN,"mörtelstein, mitte",49.358051,9.046125,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
52988,amb_000232,232,pl karl marx - pg vall d'hebron,41.440677,2.161721,0.0,Europe/Madrid,Catalonia Area de Barcelona,1
154576,de:08237:6802:0:1,NaN,wittlensweiler schule bstg 1,48.470391,8.447820,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
214765,mobiiti:stopplace:11070,NaN,stade claude boué,45.673721,-0.322552,1.0,Europe/Madrid,Nouvelle-Aquitaine Mobilités,2
169062,de:08415:29244:0:2,NaN,betzingen täleswiesenstraße,48.495144,9.151138,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
152183,de:08235:1590:0:4,NaN,"altburg, oberriedt ri. calw",48.719738,8.710927,NaN,Europe/Madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
40423,tmb_2.1413.675300,1413,av pedralbes - tòquio,41.390079,2.119079,0.0,Europe/Madrid,Autoridad de Transporte Metropolitano del Area...,0


In [39]:
# Remplacer les NaN et vides par 0
stops_df['location_type'] = stops_df['location_type'].fillna(0)

# Supprimer les espaces éventuels et convertir en entier
stops_df['location_type'] = stops_df['location_type'].astype(str).str.strip()

# Remplacer les valeurs non numériques par 0
stops_df['location_type'] = stops_df['location_type'].apply(lambda x: x if x.isdigit() else '0')

# Convertir en entier
stops_df['location_type'] = stops_df['location_type'].astype(int)

# Vérifier les valeurs uniques pour s'assurer qu'elles sont correctes
print("Valeurs uniques dans location_type :", stops_df['location_type'].unique())


Valeurs uniques dans location_type : [0]


In [40]:
# --- stop_timezone en minuscules ---
stops_df["stop_timezone"] = stops_df["stop_timezone"].str.lower()

# --- wheelchair_boarding et location_type en entier ---
stops_df["wheelchair_boarding"] = stops_df["wheelchair_boarding"].fillna(0).astype(int)
stops_df["location_type"] = stops_df["location_type"].fillna(0).astype(int)

# Vérification rapide
print(stops_df[["stop_timezone", "wheelchair_boarding", "location_type"]].head())


   stop_timezone  wheelchair_boarding  location_type
0  europe/madrid                    0              0
1  europe/madrid                    0              0
2  europe/madrid                    0              0
3  europe/madrid                    0              0
4  europe/madrid                    0              0


In [41]:
stops_df.sample(10)

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_timezone,source_folder,wheelchair_boarding
138407,de:08128:12031:0:1,NaN,"bad mergentheim, rehaklinik o.d.t.",49.496384,9.789283,0,europe/madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
113777,11b3b2a8-65dd-428a-9850-ffbc638a6b32,226,cno. bajo de huétor 83,37.156443,-3.586196,0,europe/madrid,Granada City Council (Granada city bus),2
109889,pf08086072,NaN,bellavista (pg.andalusia - rda.nord),41.622544,2.299540,0,europe/madrid,Generalitat of Catalonia (Intercity bus),0
236795,6602820,NaN,ria-sirach - ecole,42.607391,2.396965,0,europe/madrid,Réseau interurbain liO Occitanie,0
163900,de:08335:2162:0:1,NaN,glashütte sommerhoferweg,47.910283,8.909895,0,europe/madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
122766,de:06437:23155:1:1,NaN,erbach (odw.)-ebersberg b 45,49.617854,8.993780,0,europe/madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
200547,mobiiti:stopplace:57137,NaN,edmond doré,44.536869,-1.156241,0,europe/madrid,Nouvelle-Aquitaine Mobilités,2
121634,ch:23016:19482:90,NaN,jona,47.230976,8.834616,0,europe/madrid,NVBW - Nahverkehrsgesellschaft Baden-Württemb...,0
258881,sp28484,1001135,corrospedriños,42.644238,-8.275006,0,europe/madrid,Xunta de Galicia Buses,0
19952,amb_200151,200151,rotonda vallvidrera,41.417014,2.022717,0,europe/madrid,Autoridad de Transporte Metropolitano del Area...,0


In [42]:
stops_df["stop_name"] = stops_df["stop_name"].str.replace(r"\s+", " ", regex=True).str.strip()

In [43]:
stops_df.sample(10)

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_timezone,source_folder,wheelchair_boarding
40511,tmb_2.1532.687129,1532,pg bonanova - carrasco i formiguera,41.401266,2.123720,0,europe/madrid,Autoridad de Transporte Metropolitano del Area...,0
191304,mobiiti:quay:68907,ROU08R,rousseau,44.835129,-0.530032,0,europe/madrid,Nouvelle-Aquitaine Mobilités,1
217195,mobiiti:stopplace:77222,NaN,boulazac les brandes,45.168476,0.759747,0,europe/madrid,Nouvelle-Aquitaine Mobilités,1
192686,mobiiti:quay:69967,BOE37A,la boétie,44.902851,-0.701770,0,europe/madrid,Nouvelle-Aquitaine Mobilités,1
270811,sp42798,1048949,reboredo,43.209409,-8.166416,0,europe/madrid,Xunta de Galicia Buses,0
62672,gen_pf22225004,NaN,estación de tamarite de litera,41.779434,0.375951,0,europe/madrid,Catalonia Area de Barcelona,0
82761,par_8_17929,17929,av.dehesa-jaras,40.306274,-4.003662,0,europe/madrid,Consorcio Regional de Transportes de Madrid CR...,0
79933,par_8_20653,20653,honduras-est.la rambla,40.425028,-3.547973,0,europe/madrid,Consorcio Regional de Transportes de Madrid CR...,0
223039,mobiiti:quay:103266,11826A,terrien bourdil,44.963173,-0.087723,0,europe/madrid,Nouvelle-Aquitaine Mobilités,2
45337,000946,946,berenguer de palou - pont del treball,41.423688,2.191778,0,europe/madrid,Àrea Metropolitana de Barcelona (AMB),1


In [44]:
import os

# Dossier principal contenant les sous-dossiers
main_folder = "/content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN"  # mettre ton chemin exact

# Parcourir toutes les valeurs uniques de source_folder dans stops_df
for subfolder_name in stops_df["source_folder"].unique():
    subfolder_path = os.path.join(main_folder, subfolder_name)

    # Créer le dossier s'il n'existe pas
    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)

    # Sélectionner uniquement les lignes correspondant à ce sous-dossier
    df_subset = stops_df[stops_df["source_folder"] == subfolder_name].copy()

    # Supprimer la colonne source_folder
    if "source_folder" in df_subset.columns:
        df_subset.drop(columns=["source_folder"], inplace=True)

    # Définir le chemin du fichier
    output_file = os.path.join(subfolder_path, "stop_clean.txt")

    # Sauvegarder le dataframe filtré
    df_subset.to_csv(output_file, index=False, sep=",", encoding="utf-8")

    print(f"📥 Fichier créé : {output_file} ({len(df_subset)} lignes)")


📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/AISA (Bus Madrid-Aranda de Duero-Burgo de Osma)/stop_clean.txt (36 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/ALSA buses/stop_clean.txt (11470 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/AUCORSA (Autobuses de Córdoba S.A.)/stop_clean.txt (613 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/AUTNA SL/stop_clean.txt (9 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Alavabus/stop_clean.txt (678 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Alvarez Travelers Coaches/stop_clean.txt (65 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Ancebus/stop_clean.txt (26 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Auif Irunbus (Lurraldebus)/stop_clean.txt (72 lignes)
📥 Fichier créé : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN/Autocares Baraza (Baraza Coaches)/stop_clean.tx